# 3.2a Provisioning, automating and scaling the fraud endpoint

**Story:** The fraud model works, but the platform team at SafeDrive has rules: infrastructure is created
from code, endpoints run inside the company network, capacity follows the traffic, and nobody pays for idle GPUs.

**Task 3.2:** Provision and configure resources for ML and AI workloads based on existing architecture and requirements.

| Skill | What you do here | Status |
|---|---|---|
| 3.2.5 | Deploy the same model with the SageMaker SDK, Boto3 and the AWS CLI | ✅ |
| 3.2.6 | Auto scaling: target tracking, custom metric, step scaling, scheduled; how to pick the metric | ✅ |
| 3.2.10 | GPU scaling: inference component copies, scale to zero, GPU metrics | 🔒 |
| 3.2.1 | On-demand vs provisioned: serverless provisioned concurrency; Bedrock Provisioned Throughput 🔒 | ✅ |
| 3.2.2 | CloudFormation: two stacks linked by exported outputs; CDK note | ✅ |
| 3.2.4 | Endpoint in a VPC: `VpcConfig`, interface and gateway VPC endpoints | ✅ |
| 3.2.3 | Build a custom inference container and push it to Amazon ECR | 🔒 |

The sections are ordered so that only **one** instance endpoint runs at a time (kind to your quota).
Run time is about 40 minutes.

## Key points for the exam

- Hosting = `CreateModel` + `CreateEndpointConfig` + `CreateEndpoint`. The SDK `deploy()` does all three.
- Endpoint auto scaling is done by **Application Auto Scaling**: register a scalable target, then attach a policy.
- **Target tracking** is the default choice. **Step scaling** for fine control. **Scheduled** for known peaks.
- Default metric: `SageMakerVariantInvocationsPerInstance`. For FMs and slow models use concurrency
  (`ConcurrentRequestsPerModel`) because request length varies a lot.
- `VpcConfig` on the **model** puts the containers in your subnets. Private traffic to AWS services needs VPC endpoints
  (interface = PrivateLink, gateway = S3 and DynamoDB).
- CloudFormation stacks share values with `Outputs` + `Export` and `Fn::ImportValue`. An export in use cannot be deleted.
- A custom container must answer `GET /ping` and `POST /invocations` on port 8080. Images live in Amazon ECR.

| Traffic | Cheapest sensible capacity |
|---|---|
| Unknown or spiky | On-demand: serverless endpoint, Bedrock on-demand |
| Spiky but latency-sensitive | Serverless with provisioned concurrency |
| Steady and predictable | Provisioned: instances + Savings Plan, Bedrock Provisioned Throughput |

## Setup

In [ ]:
import sys; sys.path.append("..")
import json, os, shutil, subprocess, time
import pandas as pd
from botocore.exceptions import ClientError
from common import utils as U, catchup as C
os.makedirs("scripts/container", exist_ok=True)
print(U.REGION, U.bucket())

**Catch-up:** the trained fraud model (returns at once if it already exists).

In [ ]:
MODEL_DATA = C.ensure_fraud_model()
state = U.load_state()
IMAGE, ROLE, REGION, BUCKET = state["xgb_image"], U.role_arn(), U.REGION, U.bucket()
print(MODEL_DATA)
print(IMAGE)

Clients, and a sample request (5 claims from the test set, features only, as CSV).

In [ ]:
sm, smr, s3 = U.client("sagemaker"), U.client("sagemaker-runtime"), U.client("s3")
TEST_KEY = state["test_s3"].split(f"{BUCKET}/", 1)[1]
payload = U.read_csv(TEST_KEY, header=None).iloc[:5, 1:].to_csv(header=False, index=False).strip()
with open("scripts/payload.csv", "w") as f:
    f.write(payload)
print(payload.splitlines()[0])

Small helpers: "does it exist", endpoint status, and "score these CSV rows".

In [ ]:
def exists(fn, **kwargs):
    try:
        fn(**kwargs)
        return True
    except ClientError:
        return False

def endpoint_status(name):
    try:
        return sm.describe_endpoint(EndpointName=name)["EndpointStatus"]
    except ClientError:
        return None

def score(endpoint, body=payload):
    resp = smr.invoke_endpoint(EndpointName=endpoint, ContentType="text/csv", Body=body)
    text = resp["Body"].read().decode().replace("\n", ",")
    return [round(float(x), 4) for x in text.split(",") if x.strip()]

print("helpers ready")

## 3.2.5 Deploy and host models programmatically

| Tool | Level | Use it when |
|---|---|---|
| SageMaker Python SDK | High: `Model(...).deploy()` | Notebooks and data-science code |
| Boto3 | Low: one call per API | Lambda functions, automation, full control |
| AWS CLI | Low: same APIs from a shell | Shell scripts, CI/CD build steps |
| CloudFormation / CDK | Declarative | Repeatable environments (section 3.2.2) |

All of them call the same three APIs. We deploy the same model three times.

### Way 1: SageMaker Python SDK
Describe the model: image + artifact + role.

In [ ]:
import sagemaker
from sagemaker.model import Model
from sagemaker.predictor import Predictor
from sagemaker.serializers import CSVSerializer
from sagemaker.serverless import ServerlessInferenceConfig
sess = sagemaker.Session(boto_session=U.session, default_bucket=BUCKET)
SDK_NAME = "safedrive-fraud-sdk"
sdk_model = Model(image_uri=IMAGE, model_data=MODEL_DATA, role=ROLE, name=SDK_NAME,
                  sagemaker_session=sess, predictor_cls=Predictor)
print(sdk_model.name)

One `deploy()` call creates the model, the endpoint config and the endpoint, and waits (about 3 minutes, serverless).

In [ ]:
if endpoint_status(SDK_NAME) is None:
    sdk_model.deploy(endpoint_name=SDK_NAME, tags=U.TAGS,
                     serverless_inference_config=ServerlessInferenceConfig(memory_size_in_mb=2048, max_concurrency=2))
print(SDK_NAME, "->", endpoint_status(SDK_NAME))

Invoke with a `Predictor`. The serializer turns your input into the request body and sets the content type.

In [ ]:
predictor = Predictor(endpoint_name=SDK_NAME, sagemaker_session=sess, serializer=CSVSerializer())
print(predictor.predict(payload).decode().split())

### Way 2: Boto3, three explicit steps
Step 1: `create_model`.

In [ ]:
B3_NAME = "safedrive-fraud-boto3"
if not exists(sm.describe_model, ModelName=B3_NAME):
    sm.create_model(ModelName=B3_NAME, ExecutionRoleArn=ROLE, Tags=U.TAGS,
                    PrimaryContainer={"Image": IMAGE, "ModelDataUrl": MODEL_DATA})
print(sm.describe_model(ModelName=B3_NAME)["ModelArn"])

Step 2: `create_endpoint_config`. This one is a real-time endpoint on one `ml.m5.large`, because the
auto scaling section needs instances to scale.

In [ ]:
if not exists(sm.describe_endpoint_config, EndpointConfigName=B3_NAME):
    sm.create_endpoint_config(
        EndpointConfigName=B3_NAME, Tags=U.TAGS,
        ProductionVariants=[{"VariantName": "AllTraffic", "ModelName": B3_NAME,
                             "InstanceType": "ml.m5.large", "InitialInstanceCount": 1}])
print("endpoint config:", B3_NAME)

Step 3: `create_endpoint` (💰 about $0.12 per hour until deleted). It starts in the background; we wait for it later.

In [ ]:
if endpoint_status(B3_NAME) is None:
    sm.create_endpoint(EndpointName=B3_NAME, EndpointConfigName=B3_NAME, Tags=U.TAGS)
print(B3_NAME, "->", endpoint_status(B3_NAME))

### Way 3: AWS CLI
Needs the AWS CLI installed where this notebook runs. `{NAME}` in a `!` line is replaced by the Python variable.
Step 1: `create-model` (shorthand syntax for the container).

In [ ]:
CLI_NAME = "safedrive-fraud-cli"
!aws sagemaker create-model --region {REGION} --model-name {CLI_NAME} --execution-role-arn {ROLE} --primary-container Image={IMAGE},ModelDataUrl={MODEL_DATA}

Nested settings are easier in a JSON file. Write the production variants to a file.

In [ ]:
variants = [{"VariantName": "AllTraffic", "ModelName": CLI_NAME,
             "ServerlessConfig": {"MemorySizeInMB": 2048, "MaxConcurrency": 2}}]
with open("scripts/cli_variants.json", "w") as f:
    json.dump(variants, f, indent=2)
print(open("scripts/cli_variants.json").read())

Step 2: `create-endpoint-config`, reading the variants with `file://`.

In [ ]:
!aws sagemaker create-endpoint-config --region {REGION} --endpoint-config-name {CLI_NAME} --production-variants file://scripts/cli_variants.json

Step 3: `create-endpoint`.

In [ ]:
!aws sagemaker create-endpoint --region {REGION} --endpoint-name {CLI_NAME} --endpoint-config-name {CLI_NAME}

The CLI has waiters too: `wait endpoint-in-service` blocks until the endpoint is ready (about 3 minutes).

In [ ]:
!aws sagemaker wait endpoint-in-service --region {REGION} --endpoint-name {CLI_NAME}
!aws sagemaker describe-endpoint --region {REGION} --endpoint-name {CLI_NAME} --query EndpointStatus

Invoke from the CLI. `fileb://` sends the file as raw bytes; the answer is written to the last argument.

In [ ]:
!aws sagemaker-runtime invoke-endpoint --region {REGION} --endpoint-name {CLI_NAME} --content-type text/csv --body fileb://scripts/payload.csv scripts/cli_out.txt
!cat scripts/cli_out.txt

Three tools, same three resources each time. List them.

In [ ]:
rows = [{"name": n, "endpoint": endpoint_status(n),
         "config": exists(sm.describe_endpoint_config, EndpointConfigName=n),
         "model": exists(sm.describe_model, ModelName=n)} for n in [SDK_NAME, B3_NAME, CLI_NAME]]
pd.DataFrame(rows)

Delete the SDK and CLI deployments. The Boto3 endpoint stays for the next section.

In [ ]:
for name in [SDK_NAME, CLI_NAME]:
    U.safe(sm.delete_endpoint, EndpointName=name)
    U.safe(sm.delete_endpoint_config, EndpointConfigName=name)
    U.safe(sm.delete_model, ModelName=name)

## 3.2.6 Auto scaling and how to choose the metric

- Application Auto Scaling changes `DesiredInstanceCount` of a **production variant**.
- Resource ID format: `endpoint/<endpoint name>/variant/<variant name>`.

| Metric | Use it when |
|---|---|
| `SageMakerVariantInvocationsPerInstance` (predefined) | Default. Requests are similar in cost. Target = about half of the peak requests per minute one instance can handle (from a load test) |
| `CPUUtilization` / `GPUUtilization` (custom) | The model is compute-bound and request size varies |
| `ModelLatency` (custom, step scaling) | You have a latency SLA |
| `ConcurrentRequestsPerModel` (predefined, high resolution) | FMs and streaming responses; reacts in seconds |
| `ApproximateBacklogSizePerInstance` | Asynchronous endpoints (queue length) |
| `SageMakerVariantProvisionedConcurrencyUtilization` | Serverless provisioned concurrency |

First make sure the Boto3 endpoint from the last section is ready.

In [ ]:
sm.get_waiter("endpoint_in_service").wait(EndpointName=B3_NAME)
print(B3_NAME, "->", endpoint_status(B3_NAME), "| scores:", score(B3_NAME))

Register the variant as a **scalable target**: between 1 and 2 instances.

In [ ]:
aas, cw = U.client("application-autoscaling"), U.client("cloudwatch")
RESOURCE_ID = f"endpoint/{B3_NAME}/variant/AllTraffic"
DIMENSION = "sagemaker:variant:DesiredInstanceCount"
aas.register_scalable_target(ServiceNamespace="sagemaker", ResourceId=RESOURCE_ID,
                             ScalableDimension=DIMENSION, MinCapacity=1, MaxCapacity=2)
aas.describe_scalable_targets(ServiceNamespace="sagemaker", ResourceIds=[RESOURCE_ID])["ScalableTargets"]

**Target tracking** on invocations per instance: keep each instance near 100 requests per minute.
Scale out fast (60 s cooldown), scale in slowly (300 s). Look at the two CloudWatch alarms it creates for you.

In [ ]:
tt = aas.put_scaling_policy(
    PolicyName="safedrive-invocations-target", PolicyType="TargetTrackingScaling",
    ServiceNamespace="sagemaker", ResourceId=RESOURCE_ID, ScalableDimension=DIMENSION,
    TargetTrackingScalingPolicyConfiguration={
        "TargetValue": 100.0,
        "PredefinedMetricSpecification": {"PredefinedMetricType": "SageMakerVariantInvocationsPerInstance"},
        "ScaleOutCooldown": 60, "ScaleInCooldown": 300})
[a["AlarmName"] for a in tt["Alarms"]]

**Target tracking on a custom metric**: keep average CPU near 60%. Any CloudWatch metric that goes up with load
and down when you add instances can be used.

In [ ]:
DIMS = [{"Name": "EndpointName", "Value": B3_NAME}, {"Name": "VariantName", "Value": "AllTraffic"}]
aas.put_scaling_policy(
    PolicyName="safedrive-cpu-target", PolicyType="TargetTrackingScaling",
    ServiceNamespace="sagemaker", ResourceId=RESOURCE_ID, ScalableDimension=DIMENSION,
    TargetTrackingScalingPolicyConfiguration={
        "TargetValue": 60.0,
        "CustomizedMetricSpecification": {"MetricName": "CPUUtilization", "Namespace": "/aws/sagemaker/Endpoints",
                                          "Dimensions": DIMS, "Statistic": "Average"}})["PolicyARN"]

**Step scaling**: you define the steps yourself. This policy adds 1 instance; it does nothing until an alarm calls it.

In [ ]:
step = aas.put_scaling_policy(
    PolicyName="safedrive-latency-step", PolicyType="StepScaling",
    ServiceNamespace="sagemaker", ResourceId=RESOURCE_ID, ScalableDimension=DIMENSION,
    StepScalingPolicyConfiguration={
        "AdjustmentType": "ChangeInCapacity", "Cooldown": 120, "MetricAggregationType": "Average",
        "StepAdjustments": [{"MetricIntervalLowerBound": 0.0, "ScalingAdjustment": 1}]})
print(step["PolicyARN"])

Create the alarm that triggers the step policy: average `ModelLatency` above 200 ms for 2 minutes.
`ModelLatency` is in **microseconds**.

In [ ]:
LATENCY_ALARM = "safedrive-fraud-latency-high"
cw.put_metric_alarm(
    AlarmName=LATENCY_ALARM, Namespace="AWS/SageMaker", MetricName="ModelLatency", Dimensions=DIMS,
    Statistic="Average", Period=60, EvaluationPeriods=2, Threshold=200000,
    ComparisonOperator="GreaterThanThreshold", TreatMissingData="notBreaching",
    AlarmActions=[step["PolicyARN"]], Tags=U.TAGS)
cw.describe_alarms(AlarmNames=[LATENCY_ALARM])["MetricAlarms"][0]["StateValue"]

**Scheduled scaling**: change the min and max at known times (for example Monday morning claim peaks).
It is deleted with the scalable target at the end of this section.

In [ ]:
aas.put_scheduled_action(
    ServiceNamespace="sagemaker", ScheduledActionName="safedrive-weekday-morning",
    ResourceId=RESOURCE_ID, ScalableDimension=DIMENSION,
    Schedule="cron(0 8 ? * MON-FRI *)", ScalableTargetAction={"MinCapacity": 2, "MaxCapacity": 2})
actions = aas.describe_scheduled_actions(ServiceNamespace="sagemaker", ResourceId=RESOURCE_ID)
[(a["ScheduledActionName"], a["Schedule"]) for a in actions["ScheduledActions"]]

All policies on the variant.

In [ ]:
policies = aas.describe_scaling_policies(ServiceNamespace="sagemaker", ResourceId=RESOURCE_ID)["ScalingPolicies"]
pd.DataFrame(policies)[["PolicyName", "PolicyType"]]

Send load for 4 minutes (well above 100 requests per minute). Target tracking needs 3 one-minute data points
above the target before it scales out, so be patient.

In [ ]:
LOAD_SECONDS = 240
sent, start = 0, time.time()
while time.time() - start < LOAD_SECONDS:
    score(B3_NAME)
    sent += 1
print(f"sent {sent} requests in {LOAD_SECONDS} s = {sent * 60 // LOAD_SECONDS} per minute")

See what CloudWatch recorded: invocations per minute for the variant.

In [ ]:
from datetime import datetime, timedelta, timezone
now = datetime.now(timezone.utc)
stats = cw.get_metric_statistics(Namespace="AWS/SageMaker", MetricName="Invocations", Dimensions=DIMS,
                                 StartTime=now - timedelta(minutes=10), EndTime=now, Period=60, Statistics=["Sum"])
pd.DataFrame(stats["Datapoints"]).sort_values("Timestamp")[["Timestamp", "Sum"]] if stats["Datapoints"] else "no datapoints yet"

Check the scaling activities and the instance count. If the list is empty, wait a minute and run the cell again.
You should see a scale-out to 2 instances (the endpoint status becomes `Updating` for a few minutes).

In [ ]:
acts = aas.describe_scaling_activities(ServiceNamespace="sagemaker", ResourceId=RESOURCE_ID)["ScalingActivities"]
for a in acts[:3]:
    print(a["StatusCode"], "|", a["Description"], "|", a["Cause"][:110])
variant = sm.describe_endpoint(EndpointName=B3_NAME)["ProductionVariants"][0]
print("current:", variant["CurrentInstanceCount"], "desired:", variant["DesiredInstanceCount"])

Clean up this section: delete the alarm, deregister the scalable target (this removes its policies and
scheduled actions), then delete the endpoint. The model `safedrive-fraud-boto3` stays; later sections reuse it.

In [ ]:
U.safe(cw.delete_alarms, AlarmNames=[LATENCY_ALARM])
U.safe(aas.deregister_scalable_target, ServiceNamespace="sagemaker", ResourceId=RESOURCE_ID,
       ScalableDimension=DIMENSION)

In [ ]:
U.safe(sm.delete_endpoint, EndpointName=B3_NAME)
U.safe(sm.delete_endpoint_config, EndpointConfigName=B3_NAME)

## 3.2.10 AI-specific scaling for GPU workloads 🔒

- GPUs are expensive, so the goals are: **pack** models on shared GPUs, **scale on the right signal**, and **scale to zero** when idle.
- **Inference components** scale by **copy count** (`sagemaker:inference-component:DesiredCopyCount`). The endpoint adds or
  removes instances to fit the copies (`ManagedInstanceScaling`).
- **Scale to zero:** `MinInstanceCount=0` on the variant and `MinCapacity=0` for the component. Idle cost becomes zero.
- **Scale out from zero:** with zero copies, requests fail and the metric `NoCapacityInvocationFailures` rises.
  A step scaling policy on that metric brings the first copy back (takes minutes: instance start + model load).
- **Metrics:** `ConcurrentRequestsPerCopy` (high resolution, 10 s) is better than invocations for LLMs.
  `GPUUtilization` and `GPUMemoryUtilization` are in the `/aws/sagemaker/Endpoints` and `/aws/sagemaker/InferenceComponents` namespaces.
- Other GPU levers: right-size the instance family (`g5`/`g6` for inference, `p` for training), quantized models, batching, streaming.

🔒 💰 The next cells deploy an open-weight LLM on `ml.g5.2xlarge` (about $1.50 per hour) and need GPU quota for endpoints.

In [ ]:
GPU_ENDPOINT, GPU_IC = "safedrive-llm-gpu", "safedrive-llm-ic"
if U.expensive("LLM as an inference component on a GPU endpoint"):
    from sagemaker.compute_resource_requirements.resource_requirements import ResourceRequirements
    from sagemaker.enums import EndpointType
    from sagemaker.jumpstart.model import JumpStartModel
    llm = JumpStartModel(model_id="huggingface-llm-mistral-7b-instruct", role=ROLE,
                         sagemaker_session=sess, instance_type="ml.g5.2xlarge")
    llm.deploy(endpoint_name=GPU_ENDPOINT, inference_component_name=GPU_IC, accept_eula=True, tags=U.TAGS,
               endpoint_type=EndpointType.INFERENCE_COMPONENT_BASED,
               resources=ResourceRequirements(requests={"num_accelerators": 1, "memory": 16384, "copies": 1}),
               managed_instance_scaling={"MinInstanceCount": 0, "MaxInstanceCount": 2})
    print(sm.describe_inference_component(InferenceComponentName=GPU_IC)["InferenceComponentStatus"])

🔒 Register the component's **copy count** as the scalable target, with a minimum of 0.

In [ ]:
IC_RESOURCE = f"inference-component/{GPU_IC}"
IC_DIMENSION = "sagemaker:inference-component:DesiredCopyCount"
if U.expensive("scalable target for the inference component"):
    aas.register_scalable_target(ServiceNamespace="sagemaker", ResourceId=IC_RESOURCE,
                                 ScalableDimension=IC_DIMENSION, MinCapacity=0, MaxCapacity=2)
    print(aas.describe_scalable_targets(ServiceNamespace="sagemaker", ResourceIds=[IC_RESOURCE])["ScalableTargets"])

🔒 Target tracking on concurrent requests per copy: more than 4 requests in flight per copy adds a copy.
When there is no traffic, the policy scales in to 0 copies and the instance is released.

In [ ]:
if U.expensive("target tracking policy for the inference component"):
    aas.put_scaling_policy(
        PolicyName="safedrive-llm-concurrency", PolicyType="TargetTrackingScaling",
        ServiceNamespace="sagemaker", ResourceId=IC_RESOURCE, ScalableDimension=IC_DIMENSION,
        TargetTrackingScalingPolicyConfiguration={
            "TargetValue": 4.0,
            "PredefinedMetricSpecification": {
                "PredefinedMetricType": "SageMakerInferenceComponentConcurrentRequestsPerCopyHighResolution"},
            "ScaleOutCooldown": 60, "ScaleInCooldown": 600})
    print("target tracking policy attached")

🔒 Scale out **from zero**: a step policy that adds 1 copy, triggered by an alarm on `NoCapacityInvocationFailures`.

In [ ]:
ZERO_ALARM = "safedrive-llm-no-capacity"
if U.expensive("scale-out-from-zero policy and alarm"):
    from_zero = aas.put_scaling_policy(
        PolicyName="safedrive-llm-from-zero", PolicyType="StepScaling",
        ServiceNamespace="sagemaker", ResourceId=IC_RESOURCE, ScalableDimension=IC_DIMENSION,
        StepScalingPolicyConfiguration={
            "AdjustmentType": "ChangeInCapacity", "Cooldown": 60, "MetricAggregationType": "Maximum",
            "StepAdjustments": [{"MetricIntervalLowerBound": 0.0, "ScalingAdjustment": 1}]})
    cw.put_metric_alarm(
        AlarmName=ZERO_ALARM, Namespace="AWS/SageMaker", MetricName="NoCapacityInvocationFailures",
        Dimensions=[{"Name": "InferenceComponentName", "Value": GPU_IC}],
        Statistic="Maximum", Period=60, EvaluationPeriods=1, Threshold=1,
        ComparisonOperator="GreaterThanOrEqualToThreshold", TreatMissingData="missing",
        AlarmActions=[from_zero["PolicyARN"]])
    print("alarm", ZERO_ALARM, "created")

🔒 Call the LLM a few times, then read the GPU metrics that tell you if the GPU is the bottleneck.

In [ ]:
if U.expensive("invoke the LLM and read GPU metrics"):
    body = json.dumps({"inputs": "Explain a no-claims bonus in one sentence.", "parameters": {"max_new_tokens": 60}})
    for _ in range(5):
        out = smr.invoke_endpoint(EndpointName=GPU_ENDPOINT, InferenceComponentName=GPU_IC,
                                  ContentType="application/json", Body=body)
    print(out["Body"].read().decode()[:300])
    now = datetime.now(timezone.utc)
    for metric in ["GPUUtilization", "GPUMemoryUtilization"]:
        dp = cw.get_metric_statistics(
            Namespace="/aws/sagemaker/Endpoints", MetricName=metric, Statistics=["Average"], Period=60,
            Dimensions=[{"Name": "EndpointName", "Value": GPU_ENDPOINT}, {"Name": "VariantName", "Value": "AllTraffic"}],
            StartTime=now - timedelta(minutes=15), EndTime=now)["Datapoints"]
        print(metric, [round(d["Average"], 1) for d in dp])

🔒 Delete the GPU resources straight away: alarm, scalable target, component, endpoint.

In [ ]:
if U.expensive("delete the GPU endpoint"):
    U.safe(cw.delete_alarms, AlarmNames=[ZERO_ALARM])
    U.safe(aas.deregister_scalable_target, ServiceNamespace="sagemaker", ResourceId=IC_RESOURCE,
           ScalableDimension=IC_DIMENSION)
    U.safe(sm.delete_inference_component, InferenceComponentName=GPU_IC)
    time.sleep(120)
    U.safe(sm.delete_endpoint, EndpointName=GPU_ENDPOINT)
    U.safe(sm.delete_endpoint_config, EndpointConfigName=GPU_ENDPOINT)

## 3.2.1 On-demand vs provisioned resources

| Service | On-demand | Provisioned |
|---|---|---|
| SageMaker inference | Serverless endpoint (pay per ms of compute) | Instances (per hour); serverless **provisioned concurrency** (warm capacity, no cold start) |
| SageMaker training | On-demand instances | **Managed Spot** (up to 90% cheaper, can be interrupted, use checkpoints); Savings Plans |
| Bedrock | Per token | **Provisioned Throughput**: model units per hour, 1 or 6 month commitment is cheaper |

- On-demand: no commitment, good for spiky or new workloads, but cold starts or throttling can happen.
- Provisioned: predictable latency and capacity, cheaper per request only when it is busy most of the time.

Hands-on: a serverless endpoint with **provisioned concurrency** of 1 (one execution environment is kept warm).

In [ ]:
PC = "safedrive-fraud-pc"
if not exists(sm.describe_endpoint_config, EndpointConfigName=PC):
    sm.create_endpoint_config(
        EndpointConfigName=PC, Tags=U.TAGS,
        ProductionVariants=[{"VariantName": "AllTraffic", "ModelName": B3_NAME,
                             "ServerlessConfig": {"MemorySizeInMB": 2048, "MaxConcurrency": 5,
                                                  "ProvisionedConcurrency": 1}}])
print("endpoint config:", PC)

Create the endpoint and wait (about 3 minutes). 💰 Provisioned concurrency bills per second while it exists (a few cents per hour here).

In [ ]:
if endpoint_status(PC) is None:
    sm.create_endpoint(EndpointName=PC, EndpointConfigName=PC, Tags=U.TAGS)
sm.get_waiter("endpoint_in_service").wait(EndpointName=PC)
sm.describe_endpoint(EndpointName=PC)["ProductionVariants"][0]["CurrentServerlessConfig"]

Time the calls. The first call is already fast, because one environment is warm (compare with the cold start in notebook 3.1a).

In [ ]:
for i in range(4):
    start = time.time()
    score(PC)
    print(f"call {i + 1}: {(time.time() - start) * 1000:.0f} ms")

Delete the provisioned-concurrency endpoint.

In [ ]:
U.safe(sm.delete_endpoint, EndpointName=PC)
U.safe(sm.delete_endpoint_config, EndpointConfigName=PC)

### 3.2.1 Bedrock Provisioned Throughput 🔒

- You buy **model units** (a fixed number of tokens per minute) for one model.
- Needed for steady high volume and for most **custom (fine-tuned) models**.
- No commitment (hourly, most expensive), 1 month or 6 months (cheapest per hour).
- You call the model with the **provisioned model ARN** as `modelId`.

This free cell lists the models that can be bought as Provisioned Throughput in this Region.

In [ ]:
bedrock, brt = U.client("bedrock"), U.client("bedrock-runtime")
pt_models = bedrock.list_foundation_models(byInferenceType="PROVISIONED")["modelSummaries"]
pd.DataFrame(pt_models)[["modelId", "providerName"]].head(15) if pt_models else "none in this Region"

🔒 💰 Buying one model unit costs tens of dollars **per hour**, even with no commitment. Pick a `modelId` from the list above.
Leaving out `commitmentDuration` means no commitment.

In [ ]:
PT_NAME, PT_MODEL_ID = "safedrive-pt", "amazon.nova-micro-v1:0:128k"
if U.expensive("Bedrock Provisioned Throughput (billed per hour)"):
    pt_arn = bedrock.create_provisioned_model_throughput(
        provisionedModelName=PT_NAME, modelId=PT_MODEL_ID, modelUnits=1,
        tags=[{"key": "project", "value": "mla-c02-practice"}])["provisionedModelArn"]
    U.wait_until(lambda: bedrock.get_provisioned_model_throughput(provisionedModelId=pt_arn)["status"],
                 ok=("InService",), bad=("Failed",), delay=30, label="provisioned throughput")

🔒 Call the provisioned model, then delete it at once to stop the billing.

In [ ]:
if U.expensive("call and delete the Provisioned Throughput"):
    resp = brt.converse(modelId=pt_arn, inferenceConfig={"maxTokens": 50},
                        messages=[{"role": "user", "content": [{"text": "Say hello to SafeDrive."}]}])
    print(resp["output"]["message"]["content"][0]["text"])
    U.safe(bedrock.delete_provisioned_model_throughput, provisionedModelId=pt_arn)

## 3.2.2 Automate provisioning with CloudFormation (stacks that talk to each other)

- A **template** declares resources; a **stack** is one deployment of it. Updates and deletes are handled for you, with rollback on failure.
- Stacks share values with **`Outputs` + `Export`** in one stack and **`Fn::ImportValue`** in another.
  Typical split: a base stack (network, roles, model) owned by the platform team, and an application stack per endpoint.
- Other ways to pass values: nested stacks, SSM Parameter Store, and the `describe_stacks` API
  (how Step Functions, CodePipeline or a SageMaker pipeline step can read a stack output).

Stack 1 (base): the SageMaker model and an SNS topic for alerts. It **exports** the model name and topic ARN.

In [ ]:
%%writefile scripts/cfn_base.yaml
AWSTemplateFormatVersion: "2010-09-09"
Description: SafeDrive base stack - SageMaker model and alert topic, exported for other stacks
Parameters:
  ImageUri: {Type: String}
  ModelDataUrl: {Type: String}
  RoleArn: {Type: String}
Resources:
  FraudModel:
    Type: AWS::SageMaker::Model
    Properties:
      ExecutionRoleArn: !Ref RoleArn
      PrimaryContainer:
        Image: !Ref ImageUri
        ModelDataUrl: !Ref ModelDataUrl
  AlertTopic:
    Type: AWS::SNS::Topic
    Properties:
      DisplayName: SafeDrive ML alerts
Outputs:
  ModelName:
    Value: !GetAtt FraudModel.ModelName
    Export:
      Name: !Sub "${AWS::StackName}-ModelName"
  AlertTopicArn:
    Value: !Ref AlertTopic
    Export:
      Name: !Sub "${AWS::StackName}-AlertTopicArn"

Stack 2 (endpoint): **imports** the model name with `Fn::ImportValue` and creates a serverless endpoint config and the endpoint.

In [ ]:
%%writefile scripts/cfn_endpoint.yaml
AWSTemplateFormatVersion: "2010-09-09"
Description: SafeDrive endpoint stack - imports the model name from the base stack
Parameters:
  BaseStack: {Type: String}
  EndpointName: {Type: String}
Resources:
  EndpointConfig:
    Type: AWS::SageMaker::EndpointConfig
    Properties:
      ProductionVariants:
        - VariantName: AllTraffic
          InitialVariantWeight: 1
          ModelName:
            Fn::ImportValue: !Sub "${BaseStack}-ModelName"
          ServerlessConfig: {MemorySizeInMB: 2048, MaxConcurrency: 2}
  Endpoint:
    Type: AWS::SageMaker::Endpoint
    Properties:
      EndpointName: !Ref EndpointName
      EndpointConfigName: !GetAtt EndpointConfig.EndpointConfigName
Outputs:
  EndpointName: {Value: !GetAtt Endpoint.EndpointName}

Validate both templates first. This catches syntax errors and shows the parameters, without creating anything.

In [ ]:
cfn = U.client("cloudformation")
for path in ["scripts/cfn_base.yaml", "scripts/cfn_endpoint.yaml"]:
    result = cfn.validate_template(TemplateBody=open(path).read())
    print(path, "->", [p["ParameterKey"] for p in result["Parameters"]])

Create the base stack. Parameters carry the values that differ between environments.

In [ ]:
BASE_STACK, EP_STACK, CFN_ENDPOINT = "safedrive-ml-base", "safedrive-ml-endpoint", "safedrive-fraud-cfn"
if not exists(cfn.describe_stacks, StackName=BASE_STACK):
    cfn.create_stack(
        StackName=BASE_STACK, TemplateBody=open("scripts/cfn_base.yaml").read(), Tags=U.TAGS,
        Parameters=[{"ParameterKey": "ImageUri", "ParameterValue": IMAGE},
                    {"ParameterKey": "ModelDataUrl", "ParameterValue": MODEL_DATA},
                    {"ParameterKey": "RoleArn", "ParameterValue": ROLE}])
cfn.get_waiter("stack_create_complete").wait(StackName=BASE_STACK)
print(cfn.describe_stacks(StackName=BASE_STACK)["Stacks"][0]["StackStatus"])

Show the outputs of the base stack. The `ExportName` column is what other stacks import.

In [ ]:
outputs = cfn.describe_stacks(StackName=BASE_STACK)["Stacks"][0]["Outputs"]
pd.DataFrame(outputs)

Create the endpoint stack. It only gets the **name** of the base stack; CloudFormation resolves the imports (about 4 minutes).

In [ ]:
if not exists(cfn.describe_stacks, StackName=EP_STACK):
    cfn.create_stack(
        StackName=EP_STACK, TemplateBody=open("scripts/cfn_endpoint.yaml").read(), Tags=U.TAGS,
        Parameters=[{"ParameterKey": "BaseStack", "ParameterValue": BASE_STACK},
                    {"ParameterKey": "EndpointName", "ParameterValue": CFN_ENDPOINT}])
cfn.get_waiter("stack_create_complete").wait(StackName=EP_STACK)
pd.DataFrame(cfn.describe_stacks(StackName=EP_STACK)["Stacks"][0]["Outputs"])

See the link between the stacks: which stack imports the exported model name?
While this list is not empty, CloudFormation refuses to delete the base stack or change that export.

In [ ]:
print("exports:", [e["Name"] for e in cfn.list_exports()["Exports"] if e["Name"].startswith(BASE_STACK)])
print("imported by:", cfn.list_imports(ExportName=f"{BASE_STACK}-ModelName")["Imports"])

Orchestration tools read the same exports through the API. Here the notebook plays the orchestrator:
it reads the topic ARN exported by the base stack and publishes a "deployment finished" message.

In [ ]:
sns = U.client("sns")
exports = {e["Name"]: e["Value"] for e in cfn.list_exports()["Exports"]}
topic_arn = exports[f"{BASE_STACK}-AlertTopicArn"]
msg = sns.publish(TopicArn=topic_arn, Subject="SafeDrive deployment", Message=f"{CFN_ENDPOINT} is in service")
print(topic_arn, "| message id:", msg["MessageId"])

The resources the endpoint stack created, and a test call to the endpoint.

In [ ]:
resources = cfn.describe_stack_resources(StackName=EP_STACK)["StackResources"]
print(pd.DataFrame(resources)[["LogicalResourceId", "ResourceType", "ResourceStatus"]])
print("scores:", score(CFN_ENDPOINT))

Delete in the reverse order: the importing stack first.

In [ ]:
U.safe(cfn.delete_stack, StackName=EP_STACK)
cfn.get_waiter("stack_delete_complete").wait(StackName=EP_STACK)
print("deleted", EP_STACK)

In [ ]:
U.safe(cfn.delete_stack, StackName=BASE_STACK)
cfn.get_waiter("stack_delete_complete").wait(StackName=BASE_STACK)
print("deleted", BASE_STACK)

**AWS CDK (note):** you write the same infrastructure in Python or TypeScript; `cdk synth` produces a CloudFormation
template and `cdk deploy` creates the stacks. Passing an object from one stack to another creates the export and import for you.

```python
class BaseStack(Stack):
    def __init__(self, scope, id, **kw):
        super().__init__(scope, id, **kw)
        self.model = sagemaker.CfnModel(self, "FraudModel", execution_role_arn=role_arn,
                                        primary_container={"image": image, "modelDataUrl": model_data})

class EndpointStack(Stack):
    def __init__(self, scope, id, model, **kw):          # cross-stack reference
        super().__init__(scope, id, **kw)
        cfg = sagemaker.CfnEndpointConfig(self, "Config", production_variants=[{
            "variantName": "AllTraffic", "modelName": model.attr_model_name,
            "serverlessConfig": {"memorySizeInMb": 2048, "maxConcurrency": 2}}])
        sagemaker.CfnEndpoint(self, "Endpoint", endpoint_config_name=cfg.attr_endpoint_config_name)
```

## 3.2.4 SageMaker endpoints in a VPC

- `VpcConfig` (subnets + security groups) on the **model** makes SageMaker attach network interfaces (ENIs) in your subnets.
  The containers then reach private resources and follow your security group rules.
- Callers still use the SageMaker Runtime API. To keep **that** traffic private too, create an
  **interface VPC endpoint** (PrivateLink) for `sagemaker.runtime`.
- Containers without internet access need an **S3 gateway endpoint** (or NAT) to reach S3.
- `EnableNetworkIsolation=True` is stricter: the container gets no network at all.
- Use at least two subnets in different Availability Zones. Serverless endpoints do not support `VpcConfig`.
- IAM condition keys `sagemaker:VpcSubnets` and `sagemaker:VpcSecurityGroupIds` can force every model to use a VPC.

Find the default VPC.

In [ ]:
ec2 = U.client("ec2")
vpcs = ec2.describe_vpcs(Filters=[{"Name": "is-default", "Values": ["true"]}])["Vpcs"]
assert vpcs, "No default VPC in this Region. Create one with ec2.create_default_vpc() and run this cell again."
VPC_ID = vpcs[0]["VpcId"]
VPC_FILTER = [{"Name": "vpc-id", "Values": [VPC_ID]}]
print(VPC_ID, vpcs[0]["CidrBlock"])

Pick two subnets. We only use Availability Zones where the `sagemaker.runtime` interface endpoint is offered.

In [ ]:
RUNTIME_SERVICE = f"com.amazonaws.{REGION}.sagemaker.runtime"
zones = ec2.describe_vpc_endpoint_services(ServiceNames=[RUNTIME_SERVICE])["ServiceDetails"][0]["AvailabilityZones"]
subnets = [s for s in ec2.describe_subnets(Filters=VPC_FILTER)["Subnets"] if s["AvailabilityZone"] in zones]
SUBNETS = [s["SubnetId"] for s in sorted(subnets, key=lambda s: s["AvailabilityZone"])[:2]]
print(SUBNETS)

Find the default security group of the VPC and its route tables (the gateway endpoint is attached to route tables).

In [ ]:
groups = ec2.describe_security_groups(Filters=VPC_FILTER + [{"Name": "group-name", "Values": ["default"]}])
SG_ID = groups["SecurityGroups"][0]["GroupId"]
ROUTE_TABLES = [r["RouteTableId"] for r in ec2.describe_route_tables(Filters=VPC_FILTER)["RouteTables"]]
print(SG_ID, ROUTE_TABLES)

**S3 gateway endpoint** (free): adds a route to S3 that stays on the AWS network. Reuse it if the VPC already has one.

In [ ]:
S3_SERVICE = f"com.amazonaws.{REGION}.s3"
created_vpce = []
have = ec2.describe_vpc_endpoints(Filters=VPC_FILTER + [{"Name": "service-name", "Values": [S3_SERVICE]},
                                                        {"Name": "vpc-endpoint-type", "Values": ["Gateway"]}])
if have["VpcEndpoints"]:
    print("already there:", have["VpcEndpoints"][0]["VpcEndpointId"])
else:
    gw = ec2.create_vpc_endpoint(VpcEndpointType="Gateway", VpcId=VPC_ID, ServiceName=S3_SERVICE,
                                 RouteTableIds=ROUTE_TABLES,
                                 TagSpecifications=[{"ResourceType": "vpc-endpoint", "Tags": U.TAGS}])
    created_vpce.append(gw["VpcEndpoint"]["VpcEndpointId"])
    print("created:", created_vpce)

**Interface endpoint** for `sagemaker.runtime` (💰 about $0.01 per hour, deleted in cleanup). Applications in the VPC
can then invoke endpoints without leaving the AWS network. We keep private DNS off so nothing else in your VPC changes;
in production you turn it on so the normal API hostname resolves to the private address.

In [ ]:
iface = ec2.create_vpc_endpoint(
    VpcEndpointType="Interface", VpcId=VPC_ID, ServiceName=RUNTIME_SERVICE, SubnetIds=SUBNETS[:1],
    SecurityGroupIds=[SG_ID], PrivateDnsEnabled=False,
    TagSpecifications=[{"ResourceType": "vpc-endpoint", "Tags": U.TAGS}])["VpcEndpoint"]
created_vpce.append(iface["VpcEndpointId"])
print(iface["VpcEndpointId"], iface["State"])

Wait until it is available, then look at its DNS name. From inside the VPC you would call
`session.client("sagemaker-runtime", endpoint_url="https://<that DNS name>")`.

In [ ]:
def vpce_state():
    return ec2.describe_vpc_endpoints(VpcEndpointIds=[iface["VpcEndpointId"]])["VpcEndpoints"][0]["State"]

U.wait_until(vpce_state, ok=("available",), bad=("failed", "rejected"), delay=15, label="interface endpoint")
ec2.describe_vpc_endpoints(VpcEndpointIds=[iface["VpcEndpointId"]])["VpcEndpoints"][0]["DnsEntries"][0]["DnsName"]

Now the model: the same container and artifact, plus `VpcConfig`.

In [ ]:
VPC_NAME = "safedrive-fraud-vpc"
if not exists(sm.describe_model, ModelName=VPC_NAME):
    sm.create_model(ModelName=VPC_NAME, ExecutionRoleArn=ROLE, Tags=U.TAGS,
                    PrimaryContainer={"Image": IMAGE, "ModelDataUrl": MODEL_DATA},
                    VpcConfig={"SecurityGroupIds": [SG_ID], "Subnets": SUBNETS})
sm.describe_model(ModelName=VPC_NAME)["VpcConfig"]

Endpoint config and endpoint (instance based; serverless does not support a VPC model).

In [ ]:
if not exists(sm.describe_endpoint_config, EndpointConfigName=VPC_NAME):
    sm.create_endpoint_config(
        EndpointConfigName=VPC_NAME, Tags=U.TAGS,
        ProductionVariants=[{"VariantName": "AllTraffic", "ModelName": VPC_NAME,
                             "InstanceType": "ml.m5.large", "InitialInstanceCount": 1}])
if endpoint_status(VPC_NAME) is None:
    sm.create_endpoint(EndpointName=VPC_NAME, EndpointConfigName=VPC_NAME, Tags=U.TAGS)
print(VPC_NAME, "->", endpoint_status(VPC_NAME))

Wait for `InService` (about 5 to 7 minutes; attaching the network interfaces adds a little time).

In [ ]:
sm.get_waiter("endpoint_in_service").wait(EndpointName=VPC_NAME)
print(VPC_NAME, "->", endpoint_status(VPC_NAME), "| scores:", score(VPC_NAME))

Proof that the container is in your VPC: SageMaker created network interfaces in your subnets with your security group.

In [ ]:
enis = ec2.describe_network_interfaces(Filters=[{"Name": "subnet-id", "Values": SUBNETS},
                                                {"Name": "group-id", "Values": [SG_ID]}])["NetworkInterfaces"]
pd.DataFrame([{"eni": e["NetworkInterfaceId"], "subnet": e["SubnetId"], "ip": e["PrivateIpAddress"],
               "description": e.get("Description", "")[:60]} for e in enis])

Delete the VPC endpoint (SageMaker), its config and model.

In [ ]:
U.safe(sm.delete_endpoint, EndpointName=VPC_NAME)
U.safe(sm.delete_endpoint_config, EndpointConfigName=VPC_NAME)
U.safe(sm.delete_model, ModelName=VPC_NAME)

Delete the VPC endpoints this notebook created (a gateway endpoint that already existed is left alone).

In [ ]:
if created_vpce:
    U.safe(ec2.delete_vpc_endpoints, VpcEndpointIds=created_vpce)
print("deleted:", created_vpce)
created_vpce = []

## 3.2.3 Build and maintain containers 🔒

- Three choices: **built-in / prebuilt image** (least work), **extend a prebuilt image** (add libraries or code),
  **bring your own container** (any language or server).
- Inference contract: listen on port **8080**, answer `GET /ping` (health) and `POST /invocations`.
  Model files are unpacked to `/opt/ml/model`. SageMaker starts the container with `docker run <image> serve`.
- Training contract: data in `/opt/ml/input/data/<channel>`, hyperparameters in `/opt/ml/input/config`,
  write the model to `/opt/ml/model`. SageMaker runs `docker run <image> train`.
- Images are stored in **Amazon ECR**. Turn on **scan on push**, use tags or digests for versions,
  and a lifecycle policy to remove old images. Rebuild regularly to pick up security patches of the base image.

We extend the SageMaker scikit-learn image: add the `xgboost` library and bake in an inference script.
Writing the files is free; build and push are 🔒 (need Docker).

In [ ]:
%%writefile scripts/container/Dockerfile
# Extend a SageMaker prebuilt image: add a library and our inference code.
ARG BASE_IMAGE
FROM ${BASE_IMAGE}

RUN pip install --no-cache-dir xgboost==1.7.6

COPY inference.py /opt/ml/code/inference.py

# Tell the SageMaker serving toolkit which script to load.
ENV SAGEMAKER_SUBMIT_DIRECTORY=/opt/ml/code
ENV SAGEMAKER_PROGRAM=inference.py

The inference script that goes into the image. It loads the XGBoost model file from `/opt/ml/model`.

In [ ]:
%%writefile scripts/container/inference.py
"""Inference code baked into the custom image (loads the fraud XGBoost model)."""
import os

import xgboost as xgb


def model_fn(model_dir):
    booster = xgb.Booster()
    booster.load_model(os.path.join(model_dir, "xgboost-model"))
    return booster


def predict_fn(data, booster):
    return booster.predict(xgb.DMatrix(data))

Work out the image names. The base image is in an AWS-owned ECR registry; our image goes to our own registry.

In [ ]:
ecr = U.client("ecr")
REPO = "safedrive-inference"
BASE_IMAGE = sagemaker.image_uris.retrieve("sklearn", REGION, version="1.2-1", instance_type="ml.m5.large")
MY_REGISTRY = f"{U.account_id()}.dkr.ecr.{REGION}.amazonaws.com"
MY_IMAGE = f"{MY_REGISTRY}/{REPO}:v1"
DOCKER_OK = shutil.which("docker") is not None
print("docker found:", DOCKER_OK)
print("base :", BASE_IMAGE)
print("mine :", MY_IMAGE)

🔒 Create the ECR repository with scan-on-push.

In [ ]:
if DOCKER_OK and U.expensive("ECR repository for the custom image"):
    if not exists(ecr.describe_repositories, repositoryNames=[REPO]):
        ecr.create_repository(repositoryName=REPO, tags=U.TAGS,
                              imageScanningConfiguration={"scanOnPush": True})
    print(ecr.describe_repositories(repositoryNames=[REPO])["repositories"][0]["repositoryUri"])

🔒 Log Docker in to both registries. An ECR token is a temporary password (12 hours) for the user `AWS`.

In [ ]:
def docker_login(registry):
    import base64
    token = ecr.get_authorization_token()["authorizationData"][0]["authorizationToken"]
    password = base64.b64decode(token).decode().split(":", 1)[1]
    done = subprocess.run(["docker", "login", "--username", "AWS", "--password-stdin", registry],
                          input=password.encode(), capture_output=True)
    print(registry, "->", (done.stdout or done.stderr).decode().strip()[:80])

if DOCKER_OK and U.expensive("docker login to ECR"):
    docker_login(BASE_IMAGE.split("/")[0])
    docker_login(MY_REGISTRY)

🔒 Build the image (the base image is a few GB, so the first build takes some minutes).

In [ ]:
if DOCKER_OK and U.expensive("docker build"):
    !docker build --platform linux/amd64 --build-arg BASE_IMAGE={BASE_IMAGE} -t {MY_IMAGE} scripts/container
    print("built", MY_IMAGE)

🔒 Push it to ECR and look at the image record (digest, tags, scan status).

In [ ]:
if DOCKER_OK and U.expensive("docker push"):
    !docker push {MY_IMAGE}
    image = ecr.describe_images(repositoryName=REPO, imageIds=[{"imageTag": "v1"}])["imageDetails"][0]
    print(image["imageDigest"], image["imageTags"], image.get("imageScanStatus", {}).get("status"))

🔒 Deploy the fraud model with the custom image on a serverless endpoint and call it (deleted in Cleanup).

In [ ]:
BYOC = "safedrive-fraud-byoc"
if DOCKER_OK and U.expensive("deploy the custom image"):
    sm.create_model(ModelName=BYOC, ExecutionRoleArn=ROLE, Tags=U.TAGS,
                    PrimaryContainer={"Image": MY_IMAGE, "ModelDataUrl": MODEL_DATA})
    sm.create_endpoint_config(
        EndpointConfigName=BYOC, Tags=U.TAGS,
        ProductionVariants=[{"VariantName": "AllTraffic", "ModelName": BYOC,
                             "ServerlessConfig": {"MemorySizeInMB": 3072, "MaxConcurrency": 2}}])
    sm.create_endpoint(EndpointName=BYOC, EndpointConfigName=BYOC, Tags=U.TAGS)
    sm.get_waiter("endpoint_in_service").wait(EndpointName=BYOC)
    resp = smr.invoke_endpoint(EndpointName=BYOC, ContentType="text/csv", Accept="application/json", Body=payload)
    print(json.loads(resp["Body"].read()))

For a container built fully from scratch, the Dockerfile is yours end to end (notes only):

```dockerfile
FROM python:3.11-slim
RUN pip install --no-cache-dir flask gunicorn xgboost
COPY serve.py /opt/program/serve.py          # Flask app with /ping and /invocations
ENV PATH="/opt/program:${PATH}"
EXPOSE 8080
ENTRYPOINT ["gunicorn", "-b", "0.0.0.0:8080", "--chdir", "/opt/program", "serve:app"]
```

## Exam traps

- Auto scaling is configured on the **variant** through Application Auto Scaling, not in the endpoint config.
- `InvocationsPerInstance` is the default metric, but for LLMs and long requests pick a concurrency metric.
- `VpcConfig` secures the **container's** network. Private access for **callers** needs the `sagemaker.runtime` interface endpoint.
- Gateway endpoints exist only for S3 and DynamoDB and are free. Everything else is an interface endpoint (PrivateLink, hourly cost).
- You cannot delete a stack whose export is imported by another stack. Delete or update the importing stack first.

## Quiz

**1.** A real-time endpoint serves an LLM with streaming responses. Some requests take 2 seconds, others 40 seconds.
Scaling on `InvocationsPerInstance` reacts too late. Which metric is the best choice?
A. `CPUUtilization`  B. `ConcurrentRequestsPerModel`  C. `ModelSetupTime`  D. `Invocation4XXErrors`
<details><summary>Answer</summary>B. Concurrency reflects the real load when request length varies, and the high-resolution metric reacts in seconds.</details>

**2.** A network team owns a CloudFormation stack with the VPC and subnets. ML teams must create endpoints in those
subnets from their own stacks, without copying IDs by hand. What is the simplest way?
A. Hard-code the subnet IDs  B. Export the IDs as stack outputs and use `Fn::ImportValue`  C. One big stack for everyone  D. A Lambda function that edits templates
<details><summary>Answer</summary>B. Cross-stack references with exported outputs are the built-in way to share values between stacks.</details>

**3.** A model container must read from a private RDS database, and no traffic may use the public internet,
including downloads from S3. What do you configure?
A. `VpcConfig` on the model plus an S3 gateway endpoint  B. A public subnet with an internet gateway  C. `EnableNetworkIsolation` only  D. A serverless endpoint
<details><summary>Answer</summary>A. VpcConfig places the container in private subnets (reaching RDS) and the gateway endpoint gives a private path to S3. Network isolation would block RDS too.</details>

**4.** A serverless endpoint is called by a claims app during office hours. Users complain about slow first requests after quiet periods.
The team wants to stay serverless. What is the best fix?
A. Increase `MaxConcurrency`  B. Add provisioned concurrency  C. Increase memory to 6 GB  D. Move to batch transform
<details><summary>Answer</summary>B. Provisioned concurrency keeps execution environments warm and removes cold starts.</details>

**5.** An internal LLM on a GPU endpoint is used only on weekdays. The company wants zero GPU cost at night with automatic recovery when requests come back.
What should the engineer set up?
A. A multi-model endpoint  B. Inference components with min copies 0, managed instance scaling min 0, and a step policy on `NoCapacityInvocationFailures`  C. Scheduled scaling to 1 instance  D. Provisioned Throughput
<details><summary>Answer</summary>B. Scale-to-zero needs inference components; the alarm on NoCapacityInvocationFailures scales out from zero.</details>

## Save state

This notebook creates only temporary resources, so there are no new state keys. The cell shows the keys it relied on.

In [ ]:
state = U.load_state()
{k: state.get(k) for k in ["fraud_model_data", "xgb_image", "fraud_endpoint"]}

## Cleanup

**Kept for later notebooks:** nothing. Each section deleted its own resources; these cells are a safety net.

### Before you delete: what later notebooks use

**Nothing from this notebook is needed later.** It is safe to delete everything it created.

In [ ]:
for name in [SDK_NAME, B3_NAME, CLI_NAME, PC, VPC_NAME, BYOC]:
    if endpoint_status(name):
        U.safe(sm.delete_endpoint, EndpointName=name)
    if exists(sm.describe_endpoint_config, EndpointConfigName=name):
        U.safe(sm.delete_endpoint_config, EndpointConfigName=name)
    if exists(sm.describe_model, ModelName=name):
        U.safe(sm.delete_model, ModelName=name)
print("left:", [e["EndpointName"] for e in sm.list_endpoints(NameContains="safedrive-fraud-")["Endpoints"]])

Delete the latency alarm and the scalable target, in case the auto scaling section was stopped half way.

In [ ]:
U.safe(cw.delete_alarms, AlarmNames=[LATENCY_ALARM])
U.safe(aas.deregister_scalable_target, ServiceNamespace="sagemaker", ResourceId=RESOURCE_ID,
       ScalableDimension=DIMENSION)

Delete the CloudFormation stacks if they still exist (importing stack first).

In [ ]:
for stack in [EP_STACK, BASE_STACK]:
    if exists(cfn.describe_stacks, StackName=stack):
        cfn.delete_stack(StackName=stack)
        cfn.get_waiter("stack_delete_complete").wait(StackName=stack)
    print("gone:", stack)

Delete the VPC endpoints created here.

In [ ]:
if created_vpce:
    U.safe(ec2.delete_vpc_endpoints, VpcEndpointIds=created_vpce)
print("vpc endpoints:", created_vpce)

Delete the ECR repository and its images (only exists if you ran the 🔒 container cells).

In [ ]:
U.safe(ecr.delete_repository, repositoryName=REPO, force=True)